TD Final noté : Analyse des Avis et Alertes ANSSI avec
Enrichissement des CVE

● Extraire les données des flux RSS des avis et alertes ANSSI.

● Identifier les CVE mentionnées dans les bulletins.

● Enrichir les CVE avec des informations complémentaires via des API externes.


● Consolider les données dans un format exploitable (DataFrame pandas).


● Analyser et visualiser le DataFrame obtenu (vulnérabilités critiques, scores...)


● Modèles Machine learning


● Générer des alertes personnalisées pour les produits affectés et envoyer des notifications par email.

## Étape 1 — Extraction des flux RSS ANSSI

Les avis et alertes du CERT-FR sont récupérés via la fonction réutilisable
`anssi.feeds.fetch_all_feeds`. Chaque entrée est normalisée en un objet
`BulletinEntry` : **identifiant** (`CERTFR-...`), **type** (Avis / Alerte),
**titre**, **description**, **date de publication** (UTC) et **lien**.
La fonction est robuste aux champs manquants, normalise l'encodage et les
dates, et dédoublonne les bulletins.

Deux sources produisent le **même** schéma :

- **`source="rss"`** — le flux RSS live du CERT-FR ; ne renvoie que les
  bulletins les plus récents. C'est l'entrée décrite par le sujet.
- **`source="local"`** — les JSON pré-téléchargés (`data/data/`), qui couvrent
  **l'historique complet** (≈ 4 100 bulletins depuis 2021). C'est la base de
  travail des étapes d'analyse et de ML.

> **Accès responsable** (§8 du sujet) : en mode RSS, le flux brut est mis en
> cache (`data/feeds_cache/`) et relu au lieu d'être re-téléchargé, avec un
> délai de 2 s entre deux requêtes réseau.

In [ ]:
from anssi.feeds import fetch_all_feeds, feed_to_dataframe

# Flux RSS live (derniers bulletins) — la source décrite par le sujet.
recents = fetch_all_feeds(source="rss")
print(f"Flux RSS  : {len(recents)} bulletins récents")

# Corpus complet (JSON pré-téléchargés) — base de travail pour l'analyse.
bulletins = fetch_all_feeds(source="local")
print(f"Corpus local : {len(bulletins)} bulletins (avis + alertes)")

df_bulletins = feed_to_dataframe(bulletins)
df_bulletins.head()

## Étape 2 — Extraction des CVE

Chaque bulletin peut référencer des CVE dans plusieurs endroits :

| Source | Champ(s) | Fiabilité |
|---|---|---|
| Structurée | `cves[].name` | ✅ Haute — liste dédiée |
| Texte libre | `summary`, `content` | ✅ Bonne — mentions directes |
| Révisions | `revisions[].description` | ✅ Bonne — CVE ajoutés/retirés lors de mises à jour |
| Advisories tiers | `vendor_advisories[].title`, `.url` | ⚠️ Moyenne — CVE dans des URLs (risque de concaténation) |
| Systèmes affectés | `affected_systems[].description` | ⚠️ Rare |

**Validation** : les CVE sont filtrés par année (1999 ≤ year ≤ now+1) pour éliminer les faux positifs évidents (ex. `CVE-2203-23955`). Les identifiants douteux issus d'URLs (ex. `CVE-2023-2140712`) seront confirmés ou rejetés lors de l'enrichissement MITRE (étape 3).

In [ ]:
from anssi.cves import extract_all_cves, cves_to_dataframe

bulletin_cves = extract_all_cves()

total_cves = len({cve for b in bulletin_cves for cve in b.cves})
with_cves  = sum(1 for b in bulletin_cves if b.cves)
with_extra = sum(1 for b in bulletin_cves if b.cves_extra)

print(f"Bulletins traités         : {len(bulletin_cves)}")
print(f"Avec au moins 1 CVE       : {with_cves}")
print(f"Sans CVE                  : {len(bulletin_cves) - with_cves}")
print(f"Avec CVE hors clé 'cves'  : {with_extra}  (trouvés dans summary/revisions/vendor_advisories)")
print(f"CVE distincts (corpus)    : {total_cves:,}")

# Format long : une ligne par (bulletin, CVE) — base pour l'étape 4
df_cves = cves_to_dataframe(bulletin_cves)
print(f"\nDataFrame long : {df_cves.shape[0]:,} lignes × {df_cves.shape[1]} colonnes")
df_cves[df_cves["cve"].notna()].head()

## Étape 3 — Enrichissement des CVE (MITRE + FIRST/EPSS)

Chaque CVE est enrichi via deux sources :

| Source | Données | Particularités gérées |
|---|---|---|
| **MITRE** | Description, CVSS, CWE, produits/versions | 54 % sans CVSS ; priorité `v4.0 > v3.1 > v3.0 > v2.0` ; `type="cwe"` (minuscule) ; `metrics: null` ; versions avec `lessThan`/`lessThanOrEqual` |
| **FIRST** | Score EPSS, percentile | Valeurs manquantes → `None` |

Toutes les valeurs absentes sont `None` (jamais NaN) pour compatibilité pandas.

In [ ]:
from anssi.enrichment import enrich_cves, enrichment_to_dataframe

# Récupère tous les CVE uniques du corpus (étape 2)
all_cve_ids = list({cve for b in bulletin_cves for cve in b.cves})
print(f"CVE distincts à enrichir : {len(all_cve_ids):,}")

# Enrichissement depuis les données locales (§8 du sujet)
enrichments = enrich_cves(all_cve_ids, source="local")

mitre_ok = sum(1 for e in enrichments.values() if e.mitre_available)
first_ok = sum(1 for e in enrichments.values() if e.first_available)
with_cvss = sum(1 for e in enrichments.values() if e.mitre.cvss_score is not None)
with_cwe  = sum(1 for e in enrichments.values() if e.mitre.cwe_id is not None)
with_epss = sum(1 for e in enrichments.values() if e.first.epss_score is not None)

print(f"Données MITRE disponibles : {mitre_ok:,} / {len(enrichments):,}")
print(f"Données FIRST disponibles : {first_ok:,} / {len(enrichments):,}")
print(f"Avec score CVSS           : {with_cvss:,}  ({100*with_cvss/len(enrichments):.1f}%)")
print(f"Avec CWE                  : {with_cwe:,}  ({100*with_cwe/len(enrichments):.1f}%)")
print(f"Avec score EPSS           : {with_epss:,}  ({100*with_epss/len(enrichments):.1f}%)")

df_enrichment = enrichment_to_dataframe(enrichments)
df_enrichment.head()